# Module 3 - Scientific Data Manipulation
## 5. Time-based Indexing

Many scientific datasets are organised in time. Environmental and sensor-based measurements often come with timestamps, and meaningful analysis depends on treating time correctly.

In this notebook, we use date and time information to:

- convert text into datetime values,
- create a time-based index,
- filter observations by time,
- resample measurements,
- align datasets from different sources.

### Learning goals

By the end of this notebook, you should be able to:

- convert strings to datetime values,
- set a datetime index,
- select rows by time,
- resample a time series,
- align measurements collected at different timestamps,
- fill simple gaps after alignment.


## 5.1 Creating Time-stamped Data

We start with two small datasets:

- hourly air temperature,
- hourly humidity.

The timestamps are stored as text, which is common when reading CSV files.


In [1]:
import pandas as pd
import numpy as np


In [2]:
temperature_df = pd.DataFrame({
    "timestamp": ["2026-08-01 00:00", "2026-08-01 01:00", "2026-08-01 02:00", "2026-08-01 03:00"],
    "temperature": [20.1, 20.4, 20.8, 21.0]
})

humidity_df = pd.DataFrame({
    "timestamp": ["2026-08-01 00:00", "2026-08-01 01:00", "2026-08-01 02:00", "2026-08-01 03:00"],
    "humidity": [64, 63, 61, 60]
})

temperature_df


,timestamp,temperature
0,2026-08-01 00:00,20.1
1,2026-08-01 01:00,20.4
2,2026-08-01 02:00,20.8
3,2026-08-01 03:00,21.0


## 5.2 Converting to Datetime

Datetime conversion is necessary for time-based operations such as filtering by a date range, selecting by hour, or merging on time.


In [3]:
temperature_df["timestamp"] = pd.to_datetime(temperature_df["timestamp"])
humidity_df["timestamp"] = pd.to_datetime(humidity_df["timestamp"])

temperature_df


,timestamp,temperature
0,2026-08-01 00:00:00,20.1
1,2026-08-01 01:00:00,20.4
2,2026-08-01 02:00:00,20.8
3,2026-08-01 03:00:00,21.0


After conversion, the `timestamp` column is no longer plain text. It becomes a datetime type that Pandas can interpret correctly.


## 5.3 Setting a Time-based Index

A time-based index makes many operations easier and more natural.


In [4]:
temperature_df = temperature_df.set_index("timestamp")
humidity_df = humidity_df.set_index("timestamp")

temperature_df


,temperature
timestamp,
2026-08-01 00:00:00,20.1
2026-08-01 01:00:00,20.4
2026-08-01 02:00:00,20.8
2026-08-01 03:00:00,21.0


In [5]:
humidity_df


,humidity
timestamp,
2026-08-01 00:00:00,64
2026-08-01 01:00:00,63
2026-08-01 02:00:00,61
2026-08-01 03:00:00,60


When the index contains timestamps, we can filter by time using intuitive syntax.


## 5.4 Filtering by Time

Once the index contains datetime values, we can select rows by exact timestamp, by date range, or by part of a date.


In [6]:
temperature_df.loc["2026-08-01 01:00"]


temperature    20.4
Name: 2026-08-01 01:00:00, dtype: float64

In [7]:
temperature_df.loc["2026-08-01 01:00":"2026-08-01 03:00"]


,temperature
timestamp,
2026-08-01 01:00:00,20.4
2026-08-01 02:00:00,20.8
2026-08-01 03:00:00,21.0


In [8]:
temperature_df.loc["2026-08-01"]


,temperature
timestamp,
2026-08-01 00:00:00,20.1
2026-08-01 01:00:00,20.4
2026-08-01 02:00:00,20.8
2026-08-01 03:00:00,21.0


This is one of the main reasons why datetime indices are so useful in scientific work.


## 5.5 Sorting by Time

Time-based operations work best when the index is in chronological order.


In [9]:
shuffled = temperature_df.sample(frac=1, random_state=1)
print("Shuffled:")
display(shuffled)

print("Sorted again:")
display(shuffled.sort_index())


Shuffled:


,temperature
timestamp,
2026-08-01 03:00:00,21.0
2026-08-01 02:00:00,20.8
2026-08-01 00:00:00,20.1
2026-08-01 01:00:00,20.4


Sorted again:


,temperature
timestamp,
2026-08-01 00:00:00,20.1
2026-08-01 01:00:00,20.4
2026-08-01 02:00:00,20.8
2026-08-01 03:00:00,21.0


Sorting is often important after importing or merging data from external files.


## 5.6 Joining Measurements from Different Sources

If two datasets use the same timestamps, we can align them using a join.


In [10]:
combined = temperature_df.join(humidity_df, how="inner")
combined


,temperature,humidity
timestamp,,
2026-08-01 00:00:00,20.1,64
2026-08-01 01:00:00,20.4,63
2026-08-01 02:00:00,20.8,61
2026-08-01 03:00:00,21.0,60


This creates one table with temperature and humidity aligned by timestamp.


## 5.7 Resampling

Time series are often collected at one frequency but analysed at another. For example, minute-level or hourly data may later be summarised by day.


In [11]:
extended_temp = pd.DataFrame({
    "timestamp": pd.date_range("2026-08-01 00:00", periods=8, freq="6h"),
    "temperature": [20.1, 21.0, 22.2, 23.1, 21.8, 20.9, 19.7, 19.2]
}).set_index("timestamp")

extended_temp


,temperature
timestamp,
2026-08-01 00:00:00,20.1
2026-08-01 06:00:00,21.0
2026-08-01 12:00:00,22.2
2026-08-01 18:00:00,23.1
2026-08-02 00:00:00,21.8
2026-08-02 06:00:00,20.9
2026-08-02 12:00:00,19.7
2026-08-02 18:00:00,19.2


In [12]:
daily_mean = extended_temp.resample("D").mean()
daily_mean


,temperature
timestamp,
2026-08-01,21.6
2026-08-02,20.4


`resample("D")` means “group by day”. Other common frequencies include:

- `"h"` for hourly,
- `"D"` for daily,
- `"W"` for weekly,
- `"M"` for monthly.


## 5.8 Aligning Datasets with Different Timestamps

Real datasets are often not perfectly aligned. Below, temperature is measured hourly and precipitation every two hours.


In [13]:
temp = pd.DataFrame({
    "timestamp": pd.to_datetime(["2026-08-02 00:00", "2026-08-02 01:00", "2026-08-02 02:00", "2026-08-02 03:00"]),
    "temperature": [19.8, 20.0, 20.5, 20.7]
}).set_index("timestamp")

rain = pd.DataFrame({
    "timestamp": pd.to_datetime(["2026-08-02 00:00", "2026-08-02 02:00"]),
    "rain_mm": [0.0, 1.2]
}).set_index("timestamp")

print("Temperature:")
display(temp)

print("Rain:")
display(rain)


Temperature:


,temperature
timestamp,
2026-08-02 00:00:00,19.8
2026-08-02 01:00:00,20.0
2026-08-02 02:00:00,20.5
2026-08-02 03:00:00,20.7


Rain:


,rain_mm
timestamp,
2026-08-02 00:00:00,0.0
2026-08-02 02:00:00,1.2


In [14]:
aligned = temp.join(rain, how="left")
aligned


,temperature,rain_mm
timestamp,,
2026-08-02 00:00:00,19.8,0.0
2026-08-02 01:00:00,20.0,NaN
2026-08-02 02:00:00,20.5,1.2
2026-08-02 03:00:00,20.7,NaN


The rain column now contains missing values where no rain measurement was available at that exact timestamp.


## 5.9 Filling Gaps After Alignment

Once the data are aligned, we can decide how to handle the gaps. In this simple example, we use interpolation.


In [15]:
aligned_interpolated = aligned.interpolate()
aligned_interpolated


,temperature,rain_mm
timestamp,,
2026-08-02 00:00:00,19.8,0.0
2026-08-02 01:00:00,20.0,0.6
2026-08-02 02:00:00,20.5,1.2
2026-08-02 03:00:00,20.7,1.2


This is a practical workflow that appears often in scientific data preparation:

1. convert timestamps,
2. set a datetime index,
3. align multiple sources,
4. handle missing values created by alignment.


## 5.10 Resampling Before Alignment

Sometimes alignment is easier after converting both datasets to a common time frequency.


In [16]:
half_hour_temp = pd.DataFrame({
    "timestamp": pd.date_range("2026-08-03 00:00", periods=6, freq="30min"),
    "temperature": [18.8, 19.0, 19.3, 19.5, 19.8, 20.0]
}).set_index("timestamp")

hourly_temp = half_hour_temp.resample("h").mean()
hourly_temp


,temperature
timestamp,
2026-08-03 00:00:00,18.9
2026-08-03 01:00:00,19.4
2026-08-03 02:00:00,19.9


Resampling can reduce irregularity and make later merging easier.


## 5.11 Guided Practice

Complete the tasks below.

1. Convert the `timestamp` columns to datetime.
2. Set the timestamp as the index.
3. Join the two datasets.
4. Interpolate the missing rain values.


In [17]:
temp_practice = pd.DataFrame({
    "timestamp": ["2026-08-04 00:00", "2026-08-04 01:00", "2026-08-04 02:00"],
    "temperature": [19.8, 20.0, 20.5]
})

rain_practice = pd.DataFrame({
    "timestamp": ["2026-08-04 00:00", "2026-08-04 02:00"],
    "rain_mm": [0.0, 1.0]
})

# Your code here
temp_practice["timestamp"] = ...
rain_practice["timestamp"] = ...

temp_practice = ...
rain_practice = ...

joined = ...
joined_interpolated = ...

display(joined)
display(joined_interpolated)


Ellipsis

Ellipsis

### Suggested solution


In [18]:
temp_practice = pd.DataFrame({
    "timestamp": ["2026-08-04 00:00", "2026-08-04 01:00", "2026-08-04 02:00"],
    "temperature": [19.8, 20.0, 20.5]
})

rain_practice = pd.DataFrame({
    "timestamp": ["2026-08-04 00:00", "2026-08-04 02:00"],
    "rain_mm": [0.0, 1.0]
})

temp_practice["timestamp"] = pd.to_datetime(temp_practice["timestamp"])
rain_practice["timestamp"] = pd.to_datetime(rain_practice["timestamp"])

temp_practice = temp_practice.set_index("timestamp")
rain_practice = rain_practice.set_index("timestamp")

joined = temp_practice.join(rain_practice, how="left")
joined_interpolated = joined.interpolate()

display(joined)
display(joined_interpolated)


,temperature,rain_mm
timestamp,,
2026-08-04 00:00:00,19.8,0.0
2026-08-04 01:00:00,20.0,NaN
2026-08-04 02:00:00,20.5,1.0


,temperature,rain_mm
timestamp,,
2026-08-04 00:00:00,19.8,0.0
2026-08-04 01:00:00,20.0,0.5
2026-08-04 02:00:00,20.5,1.0


## 5.12 Summary

In this notebook, you learned how to:

- convert strings to datetime values,
- create and sort a time-based index,
- filter observations by time,
- join datasets on timestamps,
- resample time series,
- align sources collected at different times,
- fill simple gaps after alignment.

This concludes Module 3. Together, these notebooks introduce the main practical tools needed for basic scientific data manipulation with NumPy and Pandas.
